<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h1>Day 4 · Streaming, quality and governance</h1><p>Receive Kafka events with a persistent checkpoint; reconcile delivery and event identities; validate, quarantine and recheck a batch; document quality and governance decisions.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h1>اليوم 4 · التدفق والجودة والحوكمة</h1><p>استقبل أحداث Kafka مع نقطة تحقق مستمرة، وطابق سجلات الوصول وهويات الأحداث، وافحص الدفعة واعزل المعيب وأعد الفحص، ووثّق قرارات الجودة والحوكمة.</p></td></tr></table>



<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>1. Continue your project</h2><p>Use the same repository and successful Day 1 workspace. Restore your handoff ZIP at the repository root when using a new session. Read <a href="README.md">today’s guide</a> before running all cells in order.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>١. استكمل مشروعك</h2><p>استخدم المستودع نفسه ومساحة اليوم الأول الناجحة. استعد ملف الانتقال في جذر المستودع عند استخدام جلسة جديدة. اقرأ <a href="README.md">دليل اليوم</a> ثم شغّل الخلايا بالترتيب.</p></td></tr></table>



In [28]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'course.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the complete course repository; see docs/SETUP.md.')
sys.path.insert(0, str(ROOT / 'src'))
SOURCE = ROOT / 'data/masar-small-v1'
from masar.workspace import require_fixed_dataset, completed_bronze_workspace
from masar.runtime import require_environment, start_spark
from masar.native_contracts import validate_stage_result
require_fixed_dataset(SOURCE)
require_environment()
WORK = completed_bronze_workspace(ROOT)
print('Continue workspace:', WORK.relative_to(ROOT))


Continue workspace: outputs\day01_bronze_lk218p1f


<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>05 · Receive Kafka events</h2><p>Follow <a href="labs/lab05/WALKTHROUGH.md">the lab walkthrough</a>. The cell runs real operations and saves reports; inspect the checks and explain one observation in your lab notes.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>05 · استقبل أحداث Kafka</h2><p>اتبع <a href="labs/lab05/WALKTHROUGH.md">شرح اللاب</a>. تنفذ الخلية العمليات وتحفظ تقاريرها؛ افحص النتائج وفسّر ملاحظة واحدة في ملف اللاب.</p></td></tr></table>



In [29]:
from masar.streaming import run_stream_lab
spark = start_spark(WORK, kafka=True)
try:
    result = run_stream_lab(spark, SOURCE, WORK)
    validate_stage_result('lab05_streaming', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print('Transport rows:', [phase['transport_rows'] for phase in result['phases']])
    print('Unique event IDs:', [phase['unique_event_ids'] for phase in result['phases']])
    spark.read.format('delta').load(str(WORK/result['event_table'])).select('event_id','trip_id','event_ts').orderBy('event_id').show(5, truncate=False)
finally:
    spark.stop()


{
  "scope": "DAY04_NATIVE_STREAMING",
  "checks": {
    "phase_transport_counts": true,
    "phase_event_counts": true,
    "transport_keys_always_unique": true,
    "restart_same_query_identity": true,
    "restart_new_execution_ids": true,
    "checkpoint_same_for_all_phases": true,
    "actual_checkpoint_files_present": true,
    "producer_consumer_offsets_reconcile": true,
    "source_json_text_preserved": true,
    "event_content_matches_source": true,
    "unique_events_delta_readback": true,
    "all_events_link_to_trusted_trips": true,
    "late_event_retained": true
  }
}
Transport rows: [216, 216, 218, 219]
Unique event IDs: [216, 216, 216, 217]
+-----------+---------+-------------------------+
|event_id   |trip_id  |event_ts                 |
+-----------+---------+-------------------------+
|SYN_E0001_0|SYN_T0001|2026-06-01T06:00:00+03:00|
|SYN_E0001_1|SYN_T0001|2026-06-01T06:04:00+03:00|
|SYN_E0001_2|SYN_T0001|2026-06-01T06:08:00+03:00|
|SYN_E0002_0|SYN_T0002|2026-06-01T0

<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>06 · Validate and quarantine</h2><p>Follow <a href="labs/lab06/WALKTHROUGH.md">the lab walkthrough</a>. The cell runs real operations and saves reports; inspect the checks and explain one observation in your lab notes.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>06 · افحص واعزل السجلات</h2><p>اتبع <a href="labs/lab06/WALKTHROUGH.md">شرح اللاب</a>. تنفذ الخلية العمليات وتحفظ تقاريرها؛ افحص النتائج وفسّر ملاحظة واحدة في ملف اللاب.</p></td></tr></table>



In [30]:
from masar.quality_gate import run_quality_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_quality_lab(spark, SOURCE, WORK)
    validate_stage_result('lab06_quality', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print('Quarantined records:')
    spark.read.format('delta').load(str(WORK/result['quarantine_table'])).show(7, truncate=False)
    print('Approved rows:', spark.read.format('delta').load(str(WORK/result['approved_table'])).count())
finally:
    spark.stop()


  from .autonotebook import tqdm as notebook_tqdm



C:\Projects\masar-run\.venv\Lib\site-packages\great_expectations\data_context\store\_store_backend.py:88: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parsed_store_backend_id = store_backend_id_file_parser.parseString(


Calculating Metrics:   0%|          | 0/128 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/128 [00:00<?, ?it/s]

Calculating Metrics:   3%|▎         | 4/128 [00:00<00:00, 4133.34it/s]

...

Calculating Metrics: 100%|██████████| 128/128 [00:00<00:00, 2627.37it/s]

ERROR:great_expectations.render.renderer.site_builder:An unexpected Exception occurred during data docs rendering.  Because of this error, certain parts of data docs will not be rendered properly and/or may not appear altogether.  Please use the trace, included in this message, to diagnose and repair the underlying issue.  Detailed information follows:
                FileNotFoundError: "[Errno 2] No such file or directory: 'C:\\Projects\\masar-run\\outputs\\day01_bronze_lk218p1f\\reports\\day04_quality\\fbe6a53c8dae4843b80e9fc18fa4a6f4\\gx_rechecked\\gx\\uncommitted/data_docs/local_site/validations\\masar_silver_quality_v1\\__none__\\20260915T200520.069994Z\\masar_silver-silver_trips_snapshot.html'".  Traceback: "Traceback (most recent call last):
  File "C:\Projects\masar-run\.venv\Lib\site-packages\great_expectations\render\renderer\site_builder.py", line 476, in build
    self.target_store.set(
  File "C:\Projects\masar-run\.venv\Lib\site-packages\great_expectations\data_context\st

ERROR:great_expectations.render.renderer.site_builder:An unexpected Exception occurred during data docs rendering.  Because of this error, certain parts of data docs will not be rendered properly and/or may not appear altogether.  Please use the trace, included in this message, to diagnose and repair the underlying issue.  Detailed information follows:
                FileNotFoundError: "[Errno 2] No such file or directory: 'C:\\Projects\\masar-run\\outputs\\day01_bronze_lk218p1f\\reports\\day04_quality\\fbe6a53c8dae4843b80e9fc18fa4a6f4\\gx_rechecked\\gx\\uncommitted/data_docs/local_site/validations\\masar_silver_quality_v1\\__none__\\20260915T200520.069994Z\\masar_silver-silver_trips_snapshot.html'".  Traceback: "Traceback (most recent call last):
  File "C:\Projects\masar-run\.venv\Lib\site-packages\great_expectations\render\renderer\site_builder.py", line 476, in build
    self.target_store.set(
  File "C:\Projects\masar-run\.venv\Lib\site-packages\great_expectations\data_context\st

{
  "scope": "DAY04_NATIVE_QUALITY",
  "checks": {
    "trusted_and_rechecked_pass_gx": true,
    "mixed_candidate_fails_gx": true,
    "native_mixed_counts": true,
    "native_reasons_match_reference": true,
    "failed_candidate_not_promoted": true,
    "quarantine_delta_readback": true,
    "approved_readback_same_business_contents": true,
    "source_silver_untouched": true,
    "data_docs_exist_for_all_three_cases": true
  }
}
Quarantined records:


+-------------+----------+-------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+---------------------------------------------------------------------+
|candidate_row|trip_id   |reason_codes       |raw_business_json                                                                                                                                                                                                                                                                                                           |rule_version    |candidate_path                                                       |
+-------------+----------+-------------------+----------------------------------------------------

<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Review and save</h2><p>Answer <a href="PRACTICE.md">the questions</a> as part of the existing lab notes, then use <a href="COMPLETION.md">the completion checklist</a>. Save your notebook with actual outputs. The archive below is a handoff, not another assignment.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>راجع واحفظ</h2><p>أجب عن <a href="PRACTICE.md">الأسئلة</a> ضمن ملاحظات اللاب، ثم استخدم <a href="COMPLETION.md">قائمة الاكتمال</a>. احفظ دفترك بالمخرجات الفعلية. ملف الانتقال أدناه ليس تكليفًا آخر.</p></td></tr></table>



In [31]:
from pathlib import Path
import zipfile
pointer = ROOT / 'outputs/day01_bronze_success.json'
archive = ROOT / 'outputs/day04_handoff.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(pointer, pointer.relative_to(ROOT).as_posix())
    for path in sorted(WORK.rglob('*')):
        if path.is_file():
            bundle.write(path, path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print('Retain the notebook outputs, notes and', archive.relative_to(ROOT))


Retain the notebook outputs, notes and outputs\day04_handoff.zip
